# Sports Analysis Model Training (Colab)

Trains a sport-specific YOLO detector (rugby / football / basketball) using a
dataset pulled directly from Roboflow Universe, following the pipeline in
`MILESTONES.md` Appendix A.3–A.4.

**Before running:** get a free API key at https://app.roboflow.com/settings/api
(Workspace Settings → Roboflow API → Private API Key). You'll be prompted for
it below — it is never hardcoded into this notebook.

> Note: the `rugby` dataset config here points to `jfc/rugby-matches`
> (512 images, classes: ball/player/referee), which was verified to exist on
> Roboflow Universe. The `football`/`basketball` workspace names below
> (`roboflow/football-players`, `roboflow/basketball-detection`) are
> **unverified placeholders** — confirm them on Roboflow Universe before
> training those sports, the same way `rugby` was checked.

## 1. Setup — install packages, mount Drive, runtime check

In [1]:
#@title Install dependencies
!pip install -q roboflow ultralytics torch

import os
# Reduces GPU memory fragmentation — must be set before torch allocates
# any CUDA memory, so this goes before the import below.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 142.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.5 MB/s eta 0:00:00
CUDA available: True
Device: Tesla T4


### GPU memory utilities

`clear_gpu_memory()` runs after every major stage (download, train,
validate, export) so each stage starts from a clean slate instead of
carrying over the previous stage's tensors. `show_gpu_memory()` is safe
to call any time to check where you stand before increasing batch size
or image size.

In [ ]:
#@title GPU memory helpers
def show_gpu_memory(label: str = ""):
    if not torch.cuda.is_available():
        print(f"{label} — no CUDA device")
        return
    free_b, total_b = torch.cuda.mem_get_info()
    allocated_b = torch.cuda.memory_allocated()
    reserved_b = torch.cuda.memory_reserved()
    to_gb = lambda b: b / (1024 ** 3)
    prefix = f"[{label}] " if label else ""
    print(f"{prefix}free: {to_gb(free_b):.2f} GB / {to_gb(total_b):.2f} GB total  "
          f"(allocated: {to_gb(allocated_b):.2f} GB, reserved: {to_gb(reserved_b):.2f} GB)")


def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

show_gpu_memory("startup")

In [2]:
#@title Mount Google Drive (for checkpoints + exported models)
from google.colab import drive
drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/models"
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

Mounted at /content/drive


In [3]:
#@title Roboflow API key (prompted, never hardcoded)
import os
from getpass import getpass

ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY") or getpass("Roboflow API key: ")
os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY

Roboflow API key: ··········


## 2. Config — choose sport and training params

In [4]:
#@title Training parameters { run: "auto" }
SPORT = "rugby"  #@param ["rugby", "football", "basketball"]
EPOCHS = 100  #@param {type:"integer"}
BATCH_SIZE = 16  #@param {type:"integer"}
IMGSZ = 1280  #@param {type:"integer"}

# Treated as a starting point only — the pre-flight check in the next
# section may reduce this automatically based on free GPU memory.

In [5]:
"""
Dataset configuration per sport.

IMPORTANT: only "rugby" (jfc/rugby-matches) has been verified against
Roboflow Universe search results. Verify "football" and "basketball" the
same way at universe.roboflow.com before training them --
a wrong workspace/project will 404 partway through the run, after
packages have already been installed and Drive has already been mounted.
"""

from pathlib import Path

DATASET_CONFIGS = {
    "rugby": {
        "workspace": "jfc",
        "project": "rugby-matches",
        "version": 2,  # confirm the latest version number on the project page
        "classes": {
            0: "ball",
            1: "player",
            2: "referee",
        },
        "model": "yolov8x.pt",
    },
    "football": {
        "workspace": "roboflow",       # UNVERIFIED -- confirm before use
        "project": "football-players", # UNVERIFIED -- confirm before use
        "version": 1,
        "classes": {
            0: "player",
            1: "referee",
            2: "goalkeeper",
            32: "ball",
        },
        "model": "yolov8x.pt",
    },
    "basketball": {
        "workspace": "roboflow",           # UNVERIFIED -- confirm before use
        "project": "basketball-detection", # UNVERIFIED -- confirm before use
        "version": 1,
        "classes": {
            0: "player",
            1: "referee",
            2: "ball",
        },
        "model": "yolov8x.pt",
    },
}

dataset_config = DATASET_CONFIGS[SPORT]
print(f"Sport: {SPORT}")
print(f"  Workspace/project: {dataset_config['workspace']}/{dataset_config['project']} v{dataset_config['version']}")
print(f"  Classes: {list(dataset_config['classes'].values())}")

Sport: rugby
  Workspace/project: jfc/rugby-matches v2
  Classes: ['ball', 'player', 'referee']


## 3. Download dataset

`version.download("yolo")` fetches the full dataset (train/valid/test) in a
single call already split into folders — Roboflow's SDK does not download
per-split. The original script called `.download(f"yolo-{split}")` twice in
a loop, which just re-downloaded the identical full dataset into two
different folders rather than fetching separate splits — that loop is
removed here.

In [6]:
#@title Download dataset from Roboflow
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(dataset_config["workspace"]).project(dataset_config["project"])
version = project.version(dataset_config["version"])

dataset = version.download("yolov8")
dataset_location = Path(dataset.location)
print("Downloaded to:", dataset_location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Rugby-Matches-2 in yolov8:: 100%|██████████| 415/415 [00:00<00:00, 8890.23it/s]

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Downloaded to: /content/Rugby-Matches-2


In [7]:
#@title Write data.yaml (uses the class map from Step 2, not whatever Roboflow named them)
def create_data_yaml(output_dir: Path, classes: dict) -> Path:
    yaml_content = f"""path: {output_dir}
train: {output_dir}/train/images
val: {output_dir}/valid/images

nc: {len(classes)}
names:"""
    for idx, name in sorted(classes.items()):
        yaml_content += f"\n  {idx}: {name}"

    yaml_path = output_dir / "data.yaml"
    yaml_path.write_text(yaml_content)
    return yaml_path

yaml_path = create_data_yaml(dataset_location, dataset_config["classes"])
print("data.yaml written to:", yaml_path)
print(yaml_path.read_text())

data.yaml written to: /content/Rugby-Matches-2/data.yaml
path: /content/Rugby-Matches-2
train: /content/Rugby-Matches-2/train/images
val: /content/Rugby-Matches-2/valid/images

nc: 3
names:
  0: ball
  1: player
  2: referee


**Check before training:** open `data.yaml` above and confirm the train/val
image paths actually exist, and that the class names match what's in
Roboflow's own `data.yaml` inside the downloaded folder (`{dataset_location}/data.yaml`) —
class name mismatches won't error, they'll silently mislabel your model.

In [8]:
#@title Sanity check: compare our class map against Roboflow's own data.yaml
roboflow_yaml = dataset_location / "data.yaml"
if roboflow_yaml.exists():
    print("--- Roboflow's original data.yaml ---")
    print(roboflow_yaml.read_text())
else:
    print("No data.yaml shipped with the dataset — relying on manual class map above.")

--- Roboflow's original data.yaml ---
path: /content/Rugby-Matches-2
train: /content/Rugby-Matches-2/train/images
val: /content/Rugby-Matches-2/valid/images

nc: 3
names:
  0: ball
  1: player
  2: referee


## 3c. Pre-flight GPU memory check

Rather than let training hit CUDA OOM and auto-retry at a smaller batch
(which is what happened in the logs — `batch=16` OOM'd, retried at `8`,
OOM'd again, retried at `4`, and the retry teardown killed a
`pin_memory` worker thread each time), estimate a safe batch size
upfront from free GPU memory and only fall back to a manual, clean
retry loop if it still OOMs.

In [ ]:
#@title Estimate a safe batch size from free GPU memory
clear_gpu_memory()
show_gpu_memory("pre-training")

def estimate_safe_batch(imgsz: int, requested_batch: int, model_name: str) -> int:
    """Rough heuristic for yolov8x/l/m at a given imgsz on the free memory
    available right now. Not exact — Ultralytics' own retry is still the
    safety net — but it avoids the *first* OOM in the common case, which
    is what was tearing down dataloader workers before."""
    if not torch.cuda.is_available():
        return requested_batch

    free_b, _ = torch.cuda.mem_get_info()
    free_gb = free_b / (1024 ** 3)

    # Rough per-image memory footprint (GB) at imgsz=1280, scaled by area
    # for other sizes. Calibrated loosely from the yolov8x run in the
    # logs (batch=4 succeeded at 1280 with ~11.8GB used on a 14.9GB T4).
    per_image_gb_at_1280 = {"yolov8x.pt": 3.0, "yolov8l.pt": 2.2, "yolov8m.pt": 1.5}
    base = per_image_gb_at_1280.get(model_name, 3.0)
    scale = (imgsz / 1280) ** 2
    per_image_gb = base * scale

    headroom_gb = free_gb * 0.85  # leave 15% headroom for activations/fragmentation
    safe_batch = max(1, int(headroom_gb // per_image_gb))
    return min(requested_batch, safe_batch)

SAFE_BATCH_SIZE = estimate_safe_batch(IMGSZ, BATCH_SIZE, dataset_config["model"])
if SAFE_BATCH_SIZE < BATCH_SIZE:
    print(f"Reducing batch size {BATCH_SIZE} -> {SAFE_BATCH_SIZE} based on free GPU memory "
          f"({dataset_config['model']} @ imgsz={IMGSZ})")
else:
    print(f"Requested batch size {BATCH_SIZE} looks safe for available memory.")

## 4. Train

In [ ]:
#@title Train YOLO model (with manual OOM-safe retry)
from ultralytics import YOLO

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", device)

current_batch = SAFE_BATCH_SIZE
max_attempts = 3
results = None

for attempt in range(1, max_attempts + 1):
    clear_gpu_memory()
    show_gpu_memory(f"attempt {attempt} (batch={current_batch})")

    model = YOLO(dataset_config["model"])
    try:
        results = model.train(
            data=str(yaml_path),
            epochs=EPOCHS,
            batch=current_batch,
            imgsz=IMGSZ,
            patience=20,
            augment=True,
            mosaic=1.0,
            degrees=10.0,
            translate=0.1,
            scale=0.5,
            fliplr=0.5,
            project="/content/runs/train",
            name=f"{SPORT}-v1",
            device=device,
            amp=True,
        )
        break
    except torch.cuda.OutOfMemoryError:
        print(f"CUDA OOM at batch={current_batch}. Clearing memory and retrying at a smaller batch.")
        del model
        clear_gpu_memory()
        current_batch = max(1, current_batch // 2)
        if attempt == max_attempts:
            print("Still OOM after all retries — try a smaller model (yolov8m/l) or lower IMGSZ.")
            raise

show_gpu_memory("post-training")

Training on: cuda
Ultralytics 8.4.137 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Rugby-Matches-2/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8x.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=rugby-v

Exception in thread Thread-16 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
    ~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.13/multiprocessing/queues.py", line 120, in get
    return _ForkingPickler.loads(res)
           ~~~~~~~~~~~~~~~~~~~~~^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd
    fd = df.detach()
  File "/usr/lib/python3.13/multiprocessing/resourc


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      11.3G      1.911      10.14      1.621        124       1280: 100% ━━━━━━━━━━━━ 36/36 1.1s/it 40.8s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.6s/it 5.3s
                   all         40        983     0.0346     0.0726     0.0273    0.00933

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100      11.8G      2.272      10.02      1.818        107       1280: 100% ━━━━━━━━━━━━ 36/36 1.1s/it 38.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.7s/it 5.4s
                   all         40        983          0          0          0          0

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      11.8G      2.512      6.629      1.991         77       1280: 100% ━━━━━━━━━━━━ 36/3

## 5. Validate

In [ ]:
#@title Validation metrics
clear_gpu_memory()
show_gpu_memory("pre-validation")

metrics = model.val(data=str(yaml_path))

print(f"mAP@0.5:        {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95:   {metrics.box.map:.4f}")
print(f"Precision:      {metrics.box.p.mean():.4f}")
print(f"Recall:         {metrics.box.r.mean():.4f}")

TARGETS = {"rugby": 0.72, "football": 0.75, "basketball": 0.78}
target = TARGETS.get(SPORT)
if target is not None:
    status = "MET" if metrics.box.map >= target else "BELOW TARGET"
    print(f"\nTarget mAP@0.5:0.95 for {SPORT}: {target}  ->  {status}")

show_gpu_memory("post-validation")

## 6. Export + persist to Drive

In [ ]:
#@title Export best.pt to ONNX and copy to Drive
best_model = Path("/content/runs/train") / f"{SPORT}-v1" / "weights" / "best.pt"

# Free the training model before loading a fresh instance for export —
# otherwise both copies of a ~68M-param model sit in GPU memory at once.
del model
clear_gpu_memory()
show_gpu_memory("pre-export")

if best_model.exists():
    model_export = YOLO(str(best_model))
    onnx_path = model_export.export(format="onnx", opset=12)
    print("Exported ONNX to:", onnx_path)

    drive_export_dir = Path(CHECKPOINT_DIR) / SPORT
    drive_export_dir.mkdir(parents=True, exist_ok=True)

    import shutil
    shutil.copy(best_model, drive_export_dir / "best.pt")
    shutil.copy(onnx_path, drive_export_dir / f"{SPORT}-v1.onnx")
    print("Copied checkpoint + ONNX to:", drive_export_dir)

    del model_export
    clear_gpu_memory()
    show_gpu_memory("post-export")
else:
    print("No best.pt found at", best_model, "- training may have failed or been interrupted.")

## 7. Integration back into the project

Once validated, wire the exported ONNX into `config/{sport}.yaml` per
`MILESTONES.md` Appendix A.4:

```yaml
detection:
  model_path: "models/{sport}-v1.onnx"
  confidence_threshold: 0.35  # tune from the validation run above
```

Then verify end-to-end:

```bash
python -m src.core.pipeline --sport rugby --video data/sample_match.mp4
```

## 8. Final cleanup

Frees any remaining GPU memory. Run this before starting another
training pass (e.g. a different sport or a merged dataset) in the same
Colab session — without it, leftover allocations from this run reduce
the headroom the pre-flight check in the next run sees.

In [ ]:
#@title Release GPU memory
for var_name in ["model", "model_export", "results", "metrics"]:
    if var_name in globals():
        del globals()[var_name]

clear_gpu_memory()
show_gpu_memory("final")